# LoRA Stage 1 (all-in-one) — design HER + build the training dataset

One notebook, run top to bottom on a Colab **T4**:

1. **Cast** 20 candidate goth women (RealVisXL) -> contact sheet. Pick the
   prettiest **seed**.
2. **Lock her** — set `CHOSEN_SEED`, render the hero, cut out the background
   (-> `goth_d/body.png` for the visualizer).
3. **Bootstrap dataset** — InstantID makes ~24 varied photos of *the same face*
   (mix of portraits + full body), **filters out drifted faces** by face
   similarity, writes captions, and zips everything for Stage 2 (LoRA training).

Outputs: `her.zip` (hero body.png) and `goth_d_dataset.zip` (filtered images +
captions, trigger word `g0thd`).

> Setup is fiddly (InstantID, antelopev2, big downloads). If a cell fails, fix
> that cell and re-run — heavy downloads are cached on the runtime.

In [ ]:
# 1. GPU check.
!nvidia-smi

In [ ]:
# 2. Install. diffusers pinned for the InstantID pipeline; pillow<12 so nothing
#    breaks diffusers. (~3-4 min)
!pip install -q "diffusers==0.27.2" "huggingface_hub<0.26" transformers accelerate \
    safetensors insightface==0.7.3 onnxruntime opencv-python-headless rembg "pillow<12"
print("\n*** If pip changed Pillow: Runtime > Restart session, then run from cell 3.")

In [ ]:
# 3. Guard.
import PIL
assert int(PIL.__version__.split('.')[0]) < 12, (
    'Pillow >= 12 -> Runtime > Restart session, then run from here.')
print('Pillow', PIL.__version__, 'OK')

In [ ]:
# 4. InstantID code (repo: pipeline + ip_adapter package) + model assets.
import sys, os
if not os.path.exists('InstantID'):
    !git clone -q https://github.com/InstantID/InstantID.git
sys.path.insert(0, '/content/InstantID')

from huggingface_hub import hf_hub_download, snapshot_download
hf_hub_download('InstantX/InstantID', 'ControlNetModel/config.json', local_dir='checkpoints')
hf_hub_download('InstantX/InstantID', 'ControlNetModel/diffusion_pytorch_model.safetensors', local_dir='checkpoints')
hf_hub_download('InstantX/InstantID', 'ip-adapter.bin', local_dir='checkpoints')
snapshot_download('DIAMONIK7777/antelopev2', local_dir='models/antelopev2')
import pipeline_stable_diffusion_xl_instantid  # noqa: F401 (verify imports)
print('InstantID assets ready')

In [ ]:
# 5. Face analysis + helpers (embeddings, keypoints, identity similarity).
import cv2
import numpy as np
import torch
from insightface.app import FaceAnalysis
from pipeline_stable_diffusion_xl_instantid import draw_kps

app = FaceAnalysis(name='antelopev2', root='.', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

def biggest_face(pil_img):
    """Return the largest detected face (insightface Face) or None."""
    bgr = cv2.cvtColor(np.array(pil_img.convert('RGB')), cv2.COLOR_RGB2BGR)
    faces = app.get(bgr)
    if not faces:
        return None
    return sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))[-1]

def identity_sim(pil_img, ref_emb):
    """Cosine similarity of the image's face to the reference (or -1 if no face)."""
    f = biggest_face(pil_img)
    if f is None:
        return -1.0
    return float(np.dot(f.normed_embedding, ref_emb))

print('face analysis ready')

In [ ]:
# 6. CONFIG — edit the look, seeds, and dataset size here.
BASE_MODEL = 'SG161222/RealVisXL_V5.0'   # photoreal SDXL
CHAR_NAME = 'goth_d'
TRIGGER = 'g0thd'

# Casting / hero prompt — framing first + clothing so seeds are full-body & dressed.
FRAMING = 'full body photo, full length, head to toe, standing, wide shot'
APPEARANCE = ('goth woman, pale skin, dark grey eyes, winged eyeliner, dark lips, '
              'long black hair, blunt bangs')
OUTFIT = 'fully clothed, black gothic dress, choker, fishnet tights, platform boots'
STYLE = 'photorealistic, detailed, soft studio light, plain grey background'
HERO_PROMPT = f'{FRAMING}, {APPEARANCE}, {OUTFIT}, {STYLE}'
NEG = ('nude, naked, topless, nsfw, lingerie, close-up, portrait, headshot, cropped, '
       'extra limbs, extra fingers, bad hands, deformed, blurry, lowres, worst quality, '
       'watermark, text, anime, cartoon, 3d render, doll, child')

# 20 casting seeds.
CAST_SEEDS = [1, 7, 42, 101, 777, 2024, 3, 9, 13, 21,
              55, 88, 123, 256, 512, 999, 1234, 4321, 8888, 31337]

# Set AFTER you look at the contact sheet (cell 7).
CHOSEN_SEED = 42

# Dataset: variations for InstantID (identity fixed, look varies). 'closeup' tag
# nudges a face shot; others tend toward upper/full body.
DATASET_VARIATIONS = [
    'closeup face portrait, neutral', 'closeup face, soft smile',
    'closeup face, serious', 'closeup face, looking up',
    'upper body, head tilt', 'upper body, hand near face',
    'upper body, moody red light', 'upper body, neon purple light',
    'upper body, soft daylight', 'upper body, candlelight',
    'full body, standing', 'full body, hand on hip',
    'full body, fishnet tights', 'full body, black feather wings',
    'three quarter view, smirk', 'three quarter view, calm',
    'graveyard background, night', 'brick wall background',
    'gothic cathedral background', 'red roses background',
    'rain, wet hair', 'dramatic shadows',
    'laughing, joyful', 'eyes closed, calm',
    'side profile', 'looking over shoulder',
]
ID_THRESHOLD = 0.45   # keep images whose face matches the hero this well
print('config ready — chosen seed (edit after casting):', CHOSEN_SEED)

In [ ]:
# 7. CAST 20 candidates with plain RealVisXL. Pick a seed, set CHOSEN_SEED above.
import gc
import matplotlib.pyplot as plt
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

base_pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, variant='fp16', use_safetensors=True)
base_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    base_pipe.scheduler.config, use_karras_sigmas=True, algorithm_type='dpmsolver++')
base_pipe = base_pipe.to('cuda'); base_pipe.enable_vae_tiling()

def base_gen(seed, w=640, h=960, steps=26):
    g = torch.Generator(device='cuda').manual_seed(int(seed))
    img = base_pipe(prompt=HERO_PROMPT, negative_prompt=NEG, num_inference_steps=steps,
                    guidance_scale=5.0, width=w, height=h, generator=g).images[0]
    torch.cuda.empty_cache(); return img

cast = {s: base_gen(s) for s in CAST_SEEDS}
cols = 5; rows = (len(cast)+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(2.6*cols, 4*rows))
for ax, (s, im) in zip(axes.ravel(), cast.items()):
    ax.imshow(im); ax.set_title(f'seed {s}'); ax.axis('off')
for ax in axes.ravel()[len(cast):]: ax.axis('off')
plt.tight_layout(); plt.show()
print('>>> Pick a seed, set CHOSEN_SEED in cell 6, then run cell 8 onward.')

In [ ]:
# 8. LOCK HER — render the chosen seed big, cut the background -> hero body.png.
from pathlib import Path
from rembg import remove, new_session

hero_raw = base_gen(CHOSEN_SEED, w=768, h=1152, steps=32)
human_seg = new_session('u2net_human_seg')
hero_cut = remove(hero_raw, session=human_seg)

HERO_DIR = Path('/content/her') / CHAR_NAME; HERO_DIR.mkdir(parents=True, exist_ok=True)
hero_cut.save(HERO_DIR / 'body.png')

# Reference face embedding for InstantID + the identity filter.
ref_face = biggest_face(hero_raw)
assert ref_face is not None, 'No face in the chosen hero — pick another seed.'
ref_emb = ref_face.normed_embedding
ref_kps = draw_kps(hero_raw, ref_face.kps)

plt.figure(figsize=(4, 6)); plt.imshow(hero_cut); plt.axis('off')
plt.title(f'HER (seed {CHOSEN_SEED})'); plt.show()

# Free the casting pipeline before loading InstantID (VRAM).
del base_pipe; gc.collect(); torch.cuda.empty_cache()
print('hero saved; casting pipe freed')

In [ ]:
# 9. Load the InstantID pipeline (same RealVisXL base + IdentityNet).
from diffusers import ControlNetModel
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline

controlnet = ControlNetModel.from_pretrained('checkpoints/ControlNetModel', torch_dtype=torch.float16)
id_pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    BASE_MODEL, controlnet=controlnet, torch_dtype=torch.float16,
    use_safetensors=True, variant='fp16')
id_pipe.load_ip_adapter_instantid('checkpoints/ip-adapter.bin')
id_pipe = id_pipe.to('cuda'); id_pipe.enable_vae_tiling()
print('InstantID pipeline ready')

In [ ]:
# 10. BOOTSTRAP the dataset: generate variations, KEEP only on-identity faces.
DS = Path('/content') / f'{CHAR_NAME}_dataset'; DS.mkdir(exist_ok=True)
DBASE = (f'{TRIGGER} woman, photo of a goth woman, pale skin, dark grey eyes, '
         'dark lips, long black hair, blunt bangs, black gothic dress, '
         '{var}, photorealistic, detailed skin')

kept, shown = [], []
idx = 0
for var in DATASET_VARIATIONS:
    img = id_pipe(
        prompt=DBASE.format(var=var), negative_prompt=NEG,
        image_embeds=ref_emb, image=ref_kps,
        controlnet_conditioning_scale=0.8, ip_adapter_scale=0.8,
        num_inference_steps=30, guidance_scale=5.0, width=832, height=1024,
    ).images[0]
    torch.cuda.empty_cache()
    sim = identity_sim(img, ref_emb)
    shown.append((var, sim, img))
    if sim >= ID_THRESHOLD:
        img.save(DS / f'{idx:02d}.png')
        (DS / f'{idx:02d}.txt').write_text(f'{TRIGGER}, {var}')
        kept.append(img); idx += 1
        print(f'KEEP  sim={sim:.2f}  {var}')
    else:
        print(f'drop  sim={sim:.2f}  {var}')
print(f'\nkept {len(kept)} / {len(DATASET_VARIATIONS)} images (threshold {ID_THRESHOLD})')

In [ ]:
# 11. Preview the kept dataset (these train the LoRA in Stage 2).
if kept:
    cols = 5; rows = (len(kept)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.6*cols, 3.4*rows))
    for ax, im in zip(axes.ravel(), kept): ax.imshow(im); ax.axis('off')
    for ax in axes.ravel()[len(kept):]: ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Nothing kept — lower ID_THRESHOLD in cell 6 and re-run cell 10.')

In [ ]:
# 12. Package + download. Keep both zips for Stage 2 (LoRA training).
import shutil
from google.colab import files
shutil.make_archive('/content/her', 'zip', '/content/her')
shutil.make_archive(f'/content/{CHAR_NAME}_dataset', 'zip', str(DS))
print('Extract her.zip into assets/characters/ ; keep the dataset zip for the LoRA.')
files.download('/content/her.zip')
files.download(f'/content/{CHAR_NAME}_dataset.zip')